# Adición de nuevas estimaciones a la base de datos.

In [1]:
import pandas as pd
import numpy as np 
import os
import pickle

from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

import unicodedata

def clean_text(text):
    """Remove accents, underscores, convert to uppercase, and trim whitespace"""
    # Remove accents
    text = ''.join(c for c in unicodedata.normalize('NFD', str(text))
                   if unicodedata.category(c) != 'Mn')
    # Remove underscores
    text = text.replace('_', ' ')
    # Convert to uppercase and trim
    return text.upper().strip()

C:\Users\SANTIAGO\AppData\Roaming\Python\Python311\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
#Carga de datos anuales anteriores.
total_anual = pd.read_excel('../Data anual/data_anual_total.xlsx')
last_year =total_anual.Year.max()
year_Actual = last_year + 1
print(f"Último año registrado: {last_year}")
print(f"Año actual a predecir: {year_Actual}")

#Garantiza reescribir el año actual en caso de que ya exista en el dataset.
total_anual =total_anual[total_anual['Year'] != year_Actual]

Último año registrado: 2024
Año actual a predecir: 2025


## 1.Procesamiento de indices MODIS

In [68]:
ruta_modis_base = '../Data prod'

carpetas = os.listdir(ruta_modis_base)
carpetas_filtradas = [c for c in carpetas if os.path.isdir(os.path.join(ruta_modis_base, c))]

cols = ['Departamento', 'Municipio', 'Dataset','Year','Month', 'Date', 'Count', 'Minimum', 'Maximum' ,'Standard Deviation','Median','Mean']

total_modis = pd.DataFrame()
total_evi_aggregated = pd.DataFrame()
total_ndvi_aggregated = pd.DataFrame()


for carpeta in sorted(carpetas_filtradas):
    print(f'Procesando {carpeta}')

    depto, mpio = carpeta.split(' - ')

    mpio_corr = clean_text(mpio)
    mpio_data_hist = total_anual[total_anual['Municipio'] == mpio_corr]
    
    #Craga data MODIS
    ruta_modis = f"{ruta_modis_base}/{depto} - {mpio}/modis_{year_Actual}.csv"
    modis = pd.read_csv(ruta_modis)

   #Data total modis
    modis['Date'] = pd.to_datetime(modis['Date'])
    
    modis['YearMonth'] = modis['Date'].dt.to_period('M').astype(str)
    modis['Month'] = modis['YearMonth'].str.split('-').str[1]
    modis['Year'] = modis['YearMonth'].str.split('-').str[0].astype('int64')
    modis['Departamento'] = depto
    if mpio == 'Socorro':
        mpio = 'EL SOCORRO'
    modis['Municipio'] = mpio
    modis = modis[cols]
    modis['Departamento'] = modis['Departamento'].apply(clean_text)
    modis['Municipio'] = modis['Municipio'].apply(clean_text)


    #Data EVI
    evi = modis[modis['Dataset'] == '_500m_16_days_EVI']
    evi = evi[cols]
    evi['Date'] = pd.to_datetime(evi['Date'])

    #Obteniendo promedios por municipio
    evi_mean = mpio_data_hist.EVI_Avg_Mean.mean()
    evi_std = mpio_data_hist.EVI_Avg_StdDev_deviation.mean()

    print(f"    Promedio EVI para {mpio_corr}: {evi_mean}")
    print(f"    Desviación estándar EVI para {mpio_corr}: {evi_std}")

    #Data EVI agregando indicadores del dataset por el total del año
    evi_aggregated = evi.groupby(['Departamento', 'Municipio', 'Year']).agg({
        'Date': ['min', 'max'],
        'Count': 'max',
        'Minimum': 'min',
        'Maximum': 'max',
        'Standard Deviation': 'mean',
        'Median': 'median',
        'Mean': 'mean'
    }).reset_index()

    evi_aggregated.columns = ['Departamento', 'Municipio', 'Year', 'EVI_Min_Date', 'EVI_Max_Date', 'EVI_Max_Count', 
                                'EVI_Min_Minimum', 'EVI_Max_Maximum', 'EVI_Avg_StdDev', 'EVI_Median_Median', 'EVI_Avg_Mean']
    
    #Añadiendo marcas dedesviaciones frente promedio
    evi_aggregated['EVI_Avg_Mean_deviation'] = evi_mean - evi_aggregated['EVI_Avg_Mean']
    evi_aggregated['EVI_Avg_StdDev_deviation'] = evi_std - evi_aggregated['EVI_Avg_StdDev']
    
    #Guarda resultados en base principal 
    total_evi_aggregated = pd.concat([total_evi_aggregated,  evi_aggregated], ignore_index=True)


    #Data NDVI
    ndvi = modis[modis['Dataset'] == '_500m_16_days_NDVI']
    ndvi = ndvi[cols]
    ndvi['Date'] = pd.to_datetime(ndvi['Date'])

    ndvi_mean = mpio_data_hist.NDVI_Avg_Mean.mean()
    ndvi_std = mpio_data_hist.NDVI_Avg_StdDev_deviation.mean()

    print(f"    Promedio NDVI para {mpio_corr}: {ndvi_mean}")
    print(f"    Desviación estándar NDVI para {mpio_corr}: {ndvi_std}")

    #Data NDVI agregando indicadores del dataset por el total del año
    ndvi_aggregated = ndvi.groupby(['Departamento', 'Municipio', 'Year']).agg({
        'Date': ['min', 'max'],
        'Count': 'max',
        'Minimum': 'min',
        'Maximum': 'max',
        'Standard Deviation': 'mean',
        'Median': 'median',
        'Mean': 'mean'
    }).reset_index()

    ndvi_aggregated.columns = ['Departamento', 'Municipio', 'Year', 'NDVI_Min_Date', 'NDVI_Max_Date', 'NDVI_Max_Count', 
                                'NDVI_Min_Minimum', 'NDVI_Max_Maximum', 'NDVI_Avg_StdDev', 'NDVI_Median_Median', 'NDVI_Avg_Mean']
    ndvi_aggregated['NDVI_Avg_Mean_deviation'] = ndvi_mean - ndvi_aggregated['NDVI_Avg_Mean']
    ndvi_aggregated['NDVI_Avg_StdDev_deviation'] = ndvi_std - ndvi_aggregated['NDVI_Avg_StdDev']

    total_ndvi_aggregated = pd.concat([total_ndvi_aggregated, ndvi_aggregated], ignore_index=True)

Procesando Boyacá - Moniquirá
    Promedio EVI para MONIQUIRA: 0.4593967838164251
    Desviación estándar EVI para MONIQUIRA: -0.04435181824447597
    Promedio NDVI para MONIQUIRA: 0.7305567903381642
    Desviación estándar NDVI para MONIQUIRA: -0.05105449147120469
Procesando Boyacá - San Jose de Pare
    Promedio EVI para SAN JOSE DE PARE: 0.46171860120772945
    Desviación estándar EVI para SAN JOSE DE PARE: -0.007788269031324606
    Promedio NDVI para SAN JOSE DE PARE: 0.747719554589372
    Desviación estándar NDVI para SAN JOSE DE PARE: -0.01857659050481342
Procesando Boyacá - Santana
    Promedio EVI para SANTANA: 0.46330183309178746
    Desviación estándar EVI para SANTANA: -0.011907471276481769
    Promedio NDVI para SANTANA: 0.7499116514492753
    Desviación estándar NDVI para SANTANA: -0.019721855466385618
Procesando Boyacá - Togüí
    Promedio EVI para TOGUI: 0.44307029951690813
    Desviación estándar EVI para TOGUI: -0.04083637351544303
    Promedio NDVI para TOGUI: 0.73099

In [70]:
print(total_evi_aggregated.shape)
print(total_ndvi_aggregated.shape)

(24, 13)
(24, 13)


In [71]:
total_ndvi_aggregated

,Departamento,Municipio,Year,NDVI_Min_Date,NDVI_Max_Date,NDVI_Max_Count,NDVI_Min_Minimum,NDVI_Max_Maximum,NDVI_Avg_StdDev,NDVI_Median_Median,NDVI_Avg_Mean,NDVI_Avg_Mean_deviation,NDVI_Avg_StdDev_deviation
0,BOYACA,MONIQUIRA,2025,2025-01-01,2025-12-19,1129.0,0.0883,0.9196,0.060354,0.77460,0.757281,-0.026724,-0.111409
1,BOYACA,SAN JOSE DE PARE,2025,2025-01-01,2025-12-19,399.0,0.1238,0.8815,0.057555,0.75750,0.740989,0.006731,-0.076131
2,BOYACA,SANTANA,2025,2025-01-01,2025-12-19,392.0,0.0852,0.8863,0.063396,0.75790,0.740111,0.009800,-0.083118
3,BOYACA,TOGUI,2025,2025-01-01,2025-12-19,635.0,0.1144,0.9695,0.066131,0.76580,0.754457,-0.023464,-0.109116
4,SANTANDER,AGUADA,2025,2025-01-01,2025-12-19,339.0,0.0849,0.9062,0.065842,0.77120,0.751808,-0.025002,-0.086874
5,SANTANDER,BARBOSA,2025,2025-01-01,2025-12-19,266.0,0.1787,0.8765,0.070007,0.74265,0.725409,0.004246,-0.096398
6,SANTANDER,CHARALA,2025,2025-01-01,2025-12-19,2071.0,0.0560,0.9489,0.112713,0.79480,0.751530,-0.030757,-0.193243
7,SANTANDER,CHIPATA,2025,2025-01-01,2025-12-19,513.0,0.0621,0.9149,0.067233,0.75650,0.717959,-0.002760,-0.080886
8,SANTANDER,CONFINES,2025,2025-01-01,2025-12-19,401.0,0.0561,0.8970,0.084603,0.78180,0.740587,-0.006524,-0.107037
9,SANTANDER,CURITI,2025,2025-01-01,2025-12-19,1279.0,0.0302,0.9103,0.124065,0.70080,0.651728,-0.044749,-0.189187


## 2. Procesamiento de varibles de Copernicus

Primero se toma la información historica para calcular las desviaciones

In [72]:
data_copernicus_mes = pd.read_csv('../Data prod//ERA5_base_maestra_2007_2024_transformada.csv',sep = ';')

#Referencias temporatles
data_copernicus_mes['Mes'] = data_copernicus_mes['valid_time'].str.split('-').str[1].astype(int)
data_copernicus_mes['Year'] = data_copernicus_mes['valid_time'].str[:4].astype('int64')

#Referencias espaciales
data_copernicus_mes['departamento'] = data_copernicus_mes['departamento'].apply(clean_text)
data_copernicus_mes['municipio'] = data_copernicus_mes['municipio'].apply(clean_text)
data_copernicus_mes['municipio'] = np.where(data_copernicus_mes['municipio'] == 'EL_SOCORRO', 'SOCORRO', data_copernicus_mes['municipio'])

#Formato numérico
numeric_cols = ['latitude', 'longitude', 't2m', 'd2m', 'tp', 'ssrd', 'e', 'stl1']
data_copernicus_mes[numeric_cols] = data_copernicus_mes[numeric_cols].apply(pd.to_numeric, errors='coerce')
data_copernicus_mes.rename(columns = {'departamento': 'Departamento', 'municipio': 'Municipio'}, inplace=True)


#Promedios y desviaciones totales por municipio
copernicus_metrics_total = data_copernicus_mes.groupby(
        ['Departamento', 'Municipio']
).agg(
    t2m_mean_total=('t2m', 'mean'),
    t2m_std_total=('t2m', 'std'),
    d2m_mean_total=('d2m', 'mean'),
    d2m_std_total=('d2m', 'std'),  
    tp_mean_total=('tp', 'mean'),
    tp_std_total=('tp', 'std'),
    ssrd_mean_total=('ssrd', 'mean'),
    ssrd_std_total=('ssrd', 'std'),
    e_mean_total=('e', 'mean'),
    e_std_total=('e', 'std'), 
    stl1_mean_total=('stl1', 'mean'),
    stl1_std_total=('stl1', 'std')
).reset_index()

Luego se  actualizan las variables estimadas para este año.

In [73]:
data_copernicus_mes = pd.read_csv(f'../Data prod/ERA5_{year_Actual}_base_maestra.csv',sep = ';')
#Referencias temporatles
data_copernicus_mes['Mes'] = data_copernicus_mes['valid_time'].str.split('-').str[1].astype(int)
data_copernicus_mes['Year'] = data_copernicus_mes['valid_time'].str[:4].astype('int64')

#Referencias espaciales
data_copernicus_mes['departamento'] = data_copernicus_mes['departamento'].apply(clean_text)
data_copernicus_mes['municipio'] = data_copernicus_mes['municipio'].apply(clean_text)
data_copernicus_mes['municipio'] = np.where(data_copernicus_mes['municipio'] == 'EL_SOCORRO', 'SOCORRO', data_copernicus_mes['municipio'])

#Formato numérico
numeric_cols = ['latitude', 'longitude', 't2m', 'd2m', 'tp', 'ssrd', 'e', 'stl1']
data_copernicus_mes[numeric_cols] = data_copernicus_mes[numeric_cols].apply(pd.to_numeric, errors='coerce')
data_copernicus_mes.rename(columns = {'departamento': 'Departamento', 'municipio': 'Municipio'}, inplace=True)



In [74]:
#Agregación de datos a nivel anual
cols_Copernicus = [
       'Departamento', 'Municipio', 'Year', 'latitude_mean', 'longitude_mean',
       't2m_mean', 't2m_std', 't2m_mean_deviation', 't2m_std_deviation',
       'd2m_mean', 'd2m_std', 'd2m_mean_deviation','d2m_std_deviation',
       'tp_mean', 'tp_std', 'tp_mean_deviation','tp_std_deviation',
       'ssrd_mean', 'ssrd_std', 'ssrd_mean_deviation','ssrd_std_deviation',
       'e_mean', 'e_std', 'e_mean_deviation','e_std_deviation',
       'stl1_mean', 'stl1_std', 'stl1_mean_deviation', 'stl1_std_deviation'       
       ]

#1.Agregación del total anual
data_copernicus_aggregated = data_copernicus_mes.groupby(['Departamento', 'Municipio', 'Year']
).agg(
    latitude_mean=('latitude', 'mean'),
    longitude_mean=('longitude', 'mean'),
    t2m_mean=('t2m', 'mean'),
    t2m_std=('t2m', 'std'),
    d2m_mean=('d2m', 'mean'),
    d2m_std=('d2m', 'std'),
    tp_mean=('tp', 'mean'),
    tp_std=('tp', 'std'),
    ssrd_mean=('ssrd', 'mean'),
    ssrd_std=('ssrd', 'std'),
    e_mean=('e', 'mean'),
    e_std=('e', 'std'),
    stl1_mean=('stl1', 'mean'),
    stl1_std=('stl1', 'std')
).reset_index()

data_copernicus_aggregated = data_copernicus_aggregated.merge(copernicus_metrics_total, on=['Departamento', 'Municipio'], how='left')

data_copernicus_aggregated['t2m_mean_deviation'] = data_copernicus_aggregated['t2m_mean'] - data_copernicus_aggregated['t2m_mean_total']
data_copernicus_aggregated['t2m_std_deviation'] = data_copernicus_aggregated['t2m_std'] - data_copernicus_aggregated['t2m_std_total']
data_copernicus_aggregated['d2m_mean_deviation'] = data_copernicus_aggregated['d2m_mean'] - data_copernicus_aggregated['d2m_mean_total']
data_copernicus_aggregated['d2m_std_deviation'] = data_copernicus_aggregated['d2m_std'] - data_copernicus_aggregated['d2m_std_total']
data_copernicus_aggregated['tp_mean_deviation'] = data_copernicus_aggregated['tp_mean'] - data_copernicus_aggregated['tp_mean_total']
data_copernicus_aggregated['tp_std_deviation'] = data_copernicus_aggregated['tp_std'] - data_copernicus_aggregated['tp_std_total']
data_copernicus_aggregated['ssrd_mean_deviation'] = data_copernicus_aggregated['ssrd_mean'] - data_copernicus_aggregated['ssrd_mean_total']
data_copernicus_aggregated['ssrd_std_deviation'] = data_copernicus_aggregated['ssrd_std'] - data_copernicus_aggregated['ssrd_std_total']
data_copernicus_aggregated['e_mean_deviation'] = data_copernicus_aggregated['e_mean'] - data_copernicus_aggregated['e_mean_total']
data_copernicus_aggregated['e_std_deviation'] = data_copernicus_aggregated['e_std'] - data_copernicus_aggregated['e_std_total']
data_copernicus_aggregated['stl1_mean_deviation'] = data_copernicus_aggregated['stl1_mean'] - data_copernicus_aggregated['stl1_mean_total']
data_copernicus_aggregated['stl1_std_deviation'] = data_copernicus_aggregated['stl1_std'] - data_copernicus_aggregated['stl1_std_total']

#data_copernicus_aggregated.columns = cols_Copernicus
data_copernicus_aggregated = data_copernicus_aggregated[cols_Copernicus]

data_copernicus_aggregated.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24 entries, 0 to 23
Data columns (total 29 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Departamento         24 non-null     object 
 1   Municipio            24 non-null     object 
 2   Year                 24 non-null     int64  
 3   latitude_mean        24 non-null     float64
 4   longitude_mean       24 non-null     float64
 5   t2m_mean             24 non-null     float64
 6   t2m_std              24 non-null     float64
 7   t2m_mean_deviation   24 non-null     float64
 8   t2m_std_deviation    24 non-null     float64
 9   d2m_mean             24 non-null     float64
 10  d2m_std              24 non-null     float64
 11  d2m_mean_deviation   24 non-null     float64
 12  d2m_std_deviation    24 non-null     float64
 13  tp_mean              24 non-null     float64
 14  tp_std               24 non-null     float64
 15  tp_mean_deviation    24 non-null     float

## 3. Unificacion anual

In [75]:
merged_df = data_copernicus_aggregated.merge(total_evi_aggregated, on=['Departamento', 'Municipio', 'Year'], how='left')
merged_df = merged_df.merge(total_ndvi_aggregated, on=['Departamento', 'Municipio', 'Year'], how='left')
merged_df['Mpio'] = merged_df['Municipio'].str.strip()+'_'+merged_df['Departamento'].str.strip()
merged_df['Flag_covid'] = 0 
merged_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24 entries, 0 to 23
Data columns (total 51 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   Departamento               24 non-null     object        
 1   Municipio                  24 non-null     object        
 2   Year                       24 non-null     int64         
 3   latitude_mean              24 non-null     float64       
 4   longitude_mean             24 non-null     float64       
 5   t2m_mean                   24 non-null     float64       
 6   t2m_std                    24 non-null     float64       
 7   t2m_mean_deviation         24 non-null     float64       
 8   t2m_std_deviation          24 non-null     float64       
 9   d2m_mean                   24 non-null     float64       
 10  d2m_std                    24 non-null     float64       
 11  d2m_mean_deviation         24 non-null     float64       
 12  d2m_std_de

## 4. Estimación

In [76]:
import os

feats_select_total = [
 'EVI_Avg_Mean_deviation',
 'ssrd_mean',
 'ssrd_std',
 'EVI_Median_Median',
 'd2m_std',
 'e_std',
 'NDVI_Avg_Mean_deviation',
 'stl1_std',
 'tp_std',
 'NDVI_Max_Maximum',
 'tp_mean',
 'EVI_Max_Maximum',
 #'Flag_covid',
 'Mpio']

#Verificando que el modelo exista y cargando
print(os.path.getsize('../Artefactos/rf_model.pkl'))

with open('../Artefactos/rf_model.pkl', 'rb') as f:
    rf_model = pickle.load(f)

seed = 27

5470745


In [77]:
# -------- PREPROCESAMIENTO --------

df_pred = merged_df[feats_select_total]
cat_cols = df_pred.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = df_pred.select_dtypes(include=['int64', 'float64']).columns.tolist()


numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])
preprocessor = ColumnTransformer([
    ('num', numeric_transformer, num_cols),
    ('cat', categorical_transformer, cat_cols)
])
# -------- PIPELINE --------
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(criterion = 'squared_error', random_state=seed, n_jobs=-1))
])

#Estimación
fited_preprocessor = preprocessor.fit(df_pred)
df_pred_final = fited_preprocessor.transform(df_pred)
df_pred['Rendimiento_Predicho'] = rf_model.predict(df_pred_final)


C:\Users\SANTIAGO\AppData\Local\Temp\ipykernel_4584\1463323379.py:28: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_pred['Rendimiento_Predicho'] = rf_model.predict(df_pred_final)


## 5. Actualizacion de registro

In [85]:
#Carga de predicciones historicas
prev_red = pd.read_excel('../Data anual/total_pred_final.xlsx')
prev_red = prev_red[prev_red['Year'] != year_Actual]

In [ ]:
#Modelando y unificando el resultado de la estimacion
df_pred_save = df_pred.copy()
df_pred_save['Year'] = year_Actual
df_pred_save['Rendimiento'] = np.nan   
df_pred_save['Error'] = np.nan   
df_pred_save['Abs_Error'] = np.nan   
df_pred_save = df_pred_save[prev_red.columns]
df_pred_save = pd.concat([prev_red, df_pred_save], ignore_index=True)

In [ ]:
#Guardando resultados
df_pred_save.to_excel('../Data anual/total_pred_final.xlsx', index=False)